In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path


ROOT = Path().resolve().parent.parent

DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES        = ROOT / 'reports' / 'figures'

print('Racine projet :', ROOT)

df= pd.read_parquet(DATA_RAW / 'upgrade0.parquet')

import os
print(os.path.exists("../../data/external/upgrade0.parquet"))


df_feat = df.copy()

label_map = {
    '1A': 1, '1B': 1,
    '2A': 2, '2B': 2,
    '3A': 3, '3B': 3, '3C': 3,
    '4A': 4, '4B': 4, '4C': 4,
    '5A': 5, '5B': 5,
    '6A': 6, '6B': 6,
    '7A': 7, '7AK': 7, '7B': 7,
    '8A': 8, '8AK': 8, '8B': 8
}

if 'in.ashraeieccclimatezone2004' in df.columns:
    df['in.ashraeieccclimatezone2004'] = (
        df['in.ashraeieccclimatezone2004'].astype(str).map(label_map)
    )

drop_cols = [c for c in ['in.state', 'in.county', 'in.city', 'in.weather_file_city'] if c in df.columns]
df_feat = df_feat.drop(columns=drop_cols)

one_hot_cols = [
    'in.hvacheatingtype',
    'in.hvaccoolingtype',
    'in.heatingfuel',
    'in.geometrybuildingtyperecs',
    'in.censusregion'
]
one_hot_cols = [c for c in one_hot_cols if c in df.columns]

df_feat = pd.get_dummies(df, columns=one_hot_cols, dummy_na=False, drop_first=False)

nan_pct = df_feat.isna().mean()

drop_nan_cols = nan_pct[nan_pct > 0.30].index.tolist()
df_feat = df_feat.drop(columns=drop_nan_cols)

num_cols = df_feat.select_dtypes(include=[np.number]).columns
medians = df_feat[num_cols].median()
fill_cols = nan_pct[(nan_pct > 0) & (nan_pct < 0.05)].index.intersection(num_cols)

for c in fill_cols:
    df_feat[c] = df_feat[c].fillna(medians[c])

df_feat = df_feat.dropna()

obj_cols = df_feat.select_dtypes(include='object').columns.tolist()
assert len(obj_cols) == 0, f"Colonnes object restantes: {obj_cols}"
assert df_feat.isna().sum().sum() == 0, "NaN restants dans le dataset"

Path('data/processed').mkdir(parents=True, exist_ok=True)
df_feat.to_parquet('data/processed/metadata_features.parquet', index=False)